<a href="https://colab.research.google.com/github/mjpepito/mids-266-NLP-final/blob/main/Marvin_NLP_Final_Unified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gen Z Slang Translation + DailyDialog Action/Emotion — Unified Notebook

This notebook combines two project tracks:
1. **Utterance-level action + emotion prediction** on DailyDialog
2. **Gen Z slang ↔ regular English translation** with seq2seq baselines and improvements

It also includes an integrated pipeline that checks whether translation candidates
preserve downstream **action/emotion behavior**.

---

## Part 0 — Environment & Configuration

In [22]:
# Installs for Colab
!pip -q install transformers datasets accelerate evaluate rouge_score \
    bert_score scikit-learn pandas numpy sentencepiece torch evaluate

import os, re, ast, random, warnings
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    TrainingArguments, Trainer,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
from datasets import Dataset, DatasetDict

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch:", torch.__version__)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00
Device: cuda
Torch: 2.10.0+cu128


In [2]:
# Mount Drive -- change this for github
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── Paths ────────────────────────────────────────────────────────
# Adjust these if running outside the workspace environment.
# For Google Colab, upload the CSVs and update these paths.
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/w266/w266_final/")

# DailyDialog CSVs
DAILY_TRAIN = DATA_DIR / "train.csv"
DAILY_VAL   = DATA_DIR / "validation.csv"
DAILY_TEST  = DATA_DIR / "test.csv"
GENZ_CSV    = DATA_DIR / "GenZ_Translations.csv"

# We also pull in the larger HF Gen Z dataset directly
HF_GENZ_DATASET = "MLBtrio/genz-slang-dataset"

# ── Global config ────────────────────────────────────────────────
CONFIG = {
    "act_num_labels": 5,
    "emo_num_labels": 7,
    "context_window": 2,
    "act_epochs": 4,
    "emo_epochs": 4,
    "act_lr": 2e-5,
    "emo_lr": 2e-5,
    "batch_size": 16,
    "bart_checkpoint": "facebook/bart-base",
    "t5_checkpoint": "t5-small",
    "translation_epochs": 8,
    "translation_lr": 5e-5,
    "translation_eval_samples": 100,
    "num_translation_candidates": 8,
}

ACT_LABELS = {
    0: "dummy", 1: "inform", 2: "question",
    3: "directive", 4: "commissive"
}
EMO_LABELS = {
    0: "no_emotion", 1: "anger", 2: "disgust",
    3: "fear", 4: "happiness", 5: "sadness", 6: "surprise"
}

for name, path in [
    ("train", DAILY_TRAIN), ("val", DAILY_VAL),
    ("test", DAILY_TEST), ("genz", GENZ_CSV)
]:
    print(f"{name:>6}: {path}  exists={path.exists()}")

 train: /content/drive/MyDrive/Colab Notebooks/w266/w266_final/train.csv  exists=True
   val: /content/drive/MyDrive/Colab Notebooks/w266/w266_final/validation.csv  exists=True
  test: /content/drive/MyDrive/Colab Notebooks/w266/w266_final/test.csv  exists=True
  genz: /content/drive/MyDrive/Colab Notebooks/w266/w266_final/GenZ_Translations.csv  exists=True


---
## Part 1 — DailyDialog Preprocessing

Convert multi-turn dialogues into **single-speaker utterance rows**
with aligned action/emotion labels.

In [4]:
def _safe_parse_list(s: str) -> list:
    """Parse stringified Python list from CSV, handling edge cases."""
    s = str(s).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            return ast.literal_eval(s)
        except Exception:
            pass
    # Fallback: strip brackets, split on comma
    inner = s.strip("[] \n")
    parts = [p.strip().strip("'\"") for p in re.split(r",\s*", inner) if p.strip()]
    return parts


def parse_dialog_column(s: str) -> List[str]:
    """Extract individual utterances from the dialog column."""
    s = str(s).strip()
    utterances = _safe_parse_list(s)
    return [str(u).strip() for u in utterances if str(u).strip()]


def parse_label_column(s: str) -> List[int]:
    """Parse act or emotion label lists like '[2 1 3 2]' or '[0,0,0,0]'."""
    s = str(s).strip().strip("[]")
    parts = re.split(r"[,\s]+", s)
    return [int(p) for p in parts if p.strip()]


def load_dailydialog_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["dialog_list"] = df["dialog"].apply(parse_dialog_column)
    df["act_list"]    = df["act"].apply(parse_label_column)
    df["emotion_list"]= df["emotion"].apply(parse_label_column)
    return df


train_raw = load_dailydialog_csv(DAILY_TRAIN)
val_raw   = load_dailydialog_csv(DAILY_VAL)
test_raw  = load_dailydialog_csv(DAILY_TEST)

print(f"Train dialogues: {len(train_raw)}")
print(f"Val   dialogues: {len(val_raw)}")
print(f"Test  dialogues: {len(test_raw)}")
print("\nSample parsed dialog:", train_raw.iloc[0]["dialog_list"][:3])
print("Sample parsed acts:  ", train_raw.iloc[0]["act_list"][:3])
print("Sample parsed emos:  ", train_raw.iloc[0]["emotion_list"][:3])

Train dialogues: 11118
Val   dialogues: 1000
Test  dialogues: 1000

Sample parsed dialog: ["Say , Jim , how about going for a few beers after dinner ?  You know that is tempting but is really not good for our fitness .  What do you mean ? It will help us to relax .  Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?  I guess you are right.But what shall we do ? I don't feel like sitting at home .  I suggest a walk over to the gym where we can play singsong and meet some of our friends .  That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .  Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .  Good.Let ' s go now .  All right ."]
Sample parsed acts:   [3, 4, 2]
Sample parsed emos:   [0, 0, 0]


In [5]:
def flatten_dialogues(df: pd.DataFrame, context_size: int = 2) -> pd.DataFrame:
    """Explode each dialogue into utterance-level rows with optional context window."""
    rows = []
    for dialog_idx, row in df.iterrows():
        utts = row["dialog_list"]
        acts = row["act_list"]
        emos = row["emotion_list"]
        n = min(len(utts), len(acts), len(emos))
        for turn_idx in range(n):
            ctx_parts = []
            for j in range(max(0, turn_idx - context_size), turn_idx):
                ctx_parts.append(utts[j])
            context = " [SEP] ".join(ctx_parts) if ctx_parts else ""
            rows.append({
                "dialog_id": dialog_idx,
                "turn_id": turn_idx,
                "utterance": utts[turn_idx],
                "context": context,
                "input_text": (context + " [SEP] " + utts[turn_idx]) if context else utts[turn_idx],
                "act_label": acts[turn_idx],
                "emotion_label": emos[turn_idx],
            })
    return pd.DataFrame(rows)


train_flat = flatten_dialogues(train_raw, context_size=CONFIG["context_window"])
val_flat   = flatten_dialogues(val_raw,   context_size=CONFIG["context_window"])
test_flat  = flatten_dialogues(test_raw,  context_size=CONFIG["context_window"])

print(f"Utterance-level train: {len(train_flat)}")
print(f"Utterance-level val:   {len(val_flat)}")
print(f"Utterance-level test:  {len(test_flat)}")
print("\nAct distribution (train):")
print(train_flat["act_label"].value_counts().sort_index())
print("\nEmotion distribution (train):")
print(train_flat["emotion_label"].value_counts().sort_index())

Utterance-level train: 11118
Utterance-level val:   1000
Utterance-level test:  1000

Act distribution (train):
act_label
1    3057
2    5500
3    2557
4       4
Name: count, dtype: int64

Emotion distribution (train):
emotion_label
0    9786
1     136
2      56
3      23
4     929
5      92
6      96
Name: count, dtype: int64


---
## Part 2 — Action + Emotion Baseline (BERT)

Train utterance-level classifiers for **act** and **emotion**
using `bert-base-uncased`.

In [6]:
# ── Tokenize ─────────────────────────────────────────────────────
act_checkpoint = "bert-base-uncased"
act_tokenizer  = AutoTokenizer.from_pretrained(act_checkpoint)

def tokenize_fn(batch):
    return act_tokenizer(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )


def make_hf_dataset(df: pd.DataFrame, label_col: str) -> Dataset:
    ds = Dataset.from_pandas(
        df[["input_text", label_col]].rename(columns={label_col: "label"})
    )
    ds = ds.map(tokenize_fn, batched=True)
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return ds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": f1}

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# ── Train ACT classifier ─────────────────────────────────────────
act_model = AutoModelForSequenceClassification.from_pretrained(
    act_checkpoint, num_labels=CONFIG["act_num_labels"]
).to(DEVICE)

act_train_ds = make_hf_dataset(train_flat, "act_label")
act_val_ds   = make_hf_dataset(val_flat,   "act_label")

act_args = TrainingArguments(
    output_dir="./act_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["act_lr"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["act_epochs"],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    logging_steps=100,
    report_to="none",
)

act_trainer = Trainer(
    model=act_model,
    args=act_args,
    train_dataset=act_train_ds,
    eval_dataset=act_val_ds,
    compute_metrics=compute_metrics,
)

act_trainer.train()
act_eval = act_trainer.evaluate()
print("\nACT baseline eval:", act_eval)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/11118 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


ACT baseline eval: {'eval_loss': 0.4408564865589142, 'eval_accuracy': 0.878, 'eval_macro_f1': 0.6500322238016898, 'eval_runtime': 8.3341, 'eval_samples_per_second': 119.989, 'eval_steps_per_second': 7.559, 'epoch': 4.0}


In [8]:
# ── Train EMOTION classifier ─────────────────────────────────────
emo_model = AutoModelForSequenceClassification.from_pretrained(
    act_checkpoint, num_labels=CONFIG["emo_num_labels"]
).to(DEVICE)

emo_train_ds = make_hf_dataset(train_flat, "emotion_label")
emo_val_ds   = make_hf_dataset(val_flat,   "emotion_label")

emo_args = TrainingArguments(
    output_dir="./emo_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["emo_lr"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["emo_epochs"],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    logging_steps=100,
    report_to="none",
)

emo_trainer = Trainer(
    model=emo_model,
    args=emo_args,
    train_dataset=emo_train_ds,
    eval_dataset=emo_val_ds,
    compute_metrics=compute_metrics,
)

emo_trainer.train()
emo_eval = emo_trainer.evaluate()
print("\nEMOTION baseline eval:", emo_eval)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/11118 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.321286,0.273533,0.913000,0.287260
2,0.221737,0.252451,0.924000,0.407380
3,0.167186,0.275354,0.930000,0.430447
4,0.107298,0.300355,0.927000,0.421049


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


EMOTION baseline eval: {'eval_loss': 0.2753908634185791, 'eval_accuracy': 0.93, 'eval_macro_f1': 0.4304473023602578, 'eval_runtime': 8.3092, 'eval_samples_per_second': 120.349, 'eval_steps_per_second': 7.582, 'epoch': 4.0}


In [9]:
# ── Per-class report on test set ─────────────────────────────────
act_test_ds = make_hf_dataset(test_flat, "act_label")
emo_test_ds = make_hf_dataset(test_flat, "emotion_label")

act_preds = act_trainer.predict(act_test_ds)
emo_preds = emo_trainer.predict(emo_test_ds)

all_act_labels = list(range(CONFIG["act_num_labels"]))
all_emo_labels = list(range(CONFIG["emo_num_labels"]))

print("=== ACT Classification Report (test) ===")
print(classification_report(
    test_flat["act_label"].values,
    np.argmax(act_preds.predictions, axis=-1),
    labels=all_act_labels,
    target_names=[ACT_LABELS[i] for i in all_act_labels],
    zero_division=0,
))

print("\n=== EMOTION Classification Report (test) ===")
print(classification_report(
    test_flat["emotion_label"].values,
    np.argmax(emo_preds.predictions, axis=-1),
    labels=all_emo_labels,
    target_names=[EMO_LABELS[i] for i in all_emo_labels],
    zero_division=0,
))

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

=== ACT Classification Report (test) ===
              precision    recall  f1-score   support

       dummy       0.00      0.00      0.00         0
      inform       0.89      0.85      0.87       277
    question       0.93      0.91      0.92       497
   directive       0.76      0.85      0.80       226
  commissive       0.00      0.00      0.00         0

    accuracy                           0.88      1000
   macro avg       0.52      0.52      0.52      1000
weighted avg       0.88      0.88      0.88      1000


=== EMOTION Classification Report (test) ===
              precision    recall  f1-score   support

  no_emotion       0.94      0.95      0.94       867
       anger       0.43      0.56      0.49        16
     disgust       0.00      0.00      0.00         6
        fear       1.00      0.33      0.50         3
   happiness       0.65      0.58      0.61        92
     sadness       0.29      0.25      0.27         8
    surprise       0.25      0.25      0.25  

---
## Part 2B — Action + Emotion Improvement (BERTweet)

BERTweet is pre-trained on English tweets and tends to handle informal/short
text better than vanilla BERT. We use it as an improved encoder for both tasks.

In [10]:
# ── Improved model using BERTweet ────────────────────────────────
bertweet_checkpoint = "vinai/bertweet-base"
bertweet_tokenizer  = AutoTokenizer.from_pretrained(
    bertweet_checkpoint, use_fast=False
)


def tokenize_bertweet(batch):
    return bertweet_tokenizer(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )


def make_bertweet_dataset(df: pd.DataFrame, label_col: str) -> Dataset:
    ds = Dataset.from_pandas(
        df[["input_text", label_col]].rename(columns={label_col: "label"})
    )
    ds = ds.map(tokenize_bertweet, batched=True)
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return ds


# ACT with BERTweet
act_bt_model = AutoModelForSequenceClassification.from_pretrained(
    bertweet_checkpoint, num_labels=CONFIG["act_num_labels"]
).to(DEVICE)

act_bt_train = make_bertweet_dataset(train_flat, "act_label")
act_bt_val   = make_bertweet_dataset(val_flat,   "act_label")

act_bt_args = TrainingArguments(
    output_dir="./act_bertweet_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["act_lr"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["act_epochs"],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    logging_steps=100,
    report_to="none",
)

act_bt_trainer = Trainer(
    model=act_bt_model, args=act_bt_args,
    train_dataset=act_bt_train, eval_dataset=act_bt_val,
    compute_metrics=compute_metrics,
)
act_bt_trainer.train()
act_bt_eval = act_bt_trainer.evaluate()
print("ACT BERTweet eval:", act_bt_eval)


# EMOTION with BERTweet
emo_bt_model = AutoModelForSequenceClassification.from_pretrained(
    bertweet_checkpoint, num_labels=CONFIG["emo_num_labels"]
).to(DEVICE)

emo_bt_train = make_bertweet_dataset(train_flat, "emotion_label")
emo_bt_val   = make_bertweet_dataset(val_flat,   "emotion_label")

emo_bt_args = TrainingArguments(
    output_dir="./emo_bertweet_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["emo_lr"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["emo_epochs"],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    logging_steps=100,
    report_to="none",
)

emo_bt_trainer = Trainer(
    model=emo_bt_model, args=emo_bt_args,
    train_dataset=emo_bt_train, eval_dataset=emo_bt_val,
    compute_metrics=compute_metrics,
)
emo_bt_trainer.train()
emo_bt_eval = emo_bt_trainer.evaluate()
print("EMOTION BERTweet eval:", emo_bt_eval)

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

EMOTION BERTweet eval: {'eval_loss': 0.2956432104110718, 'eval_accuracy': 0.93, 'eval_macro_f1': 0.3282616712277826, 'eval_runtime': 7.8828, 'eval_samples_per_second': 126.859, 'eval_steps_per_second': 7.992, 'epoch': 4.0}


In [11]:
# ── Compare BERT vs BERTweet ─────────────────────────────────────
comparison = pd.DataFrame([
    {"model": "bert-base", "task": "act",     **act_eval},
    {"model": "bertweet",  "task": "act",     **act_bt_eval},
    {"model": "bert-base", "task": "emotion", **emo_eval},
    {"model": "bertweet",  "task": "emotion", **emo_bt_eval},
])
display(comparison[["model", "task", "eval_accuracy", "eval_macro_f1"]])

,model,task,eval_accuracy,eval_macro_f1
0,bert-base,act,0.878,0.650032
1,bertweet,act,0.887,0.655839
2,bert-base,emotion,0.930,0.430447
3,bertweet,emotion,0.930,0.328262


In [12]:
# ── Inference helper: predict action + emotion for any text ──────
def predict_act_emotion(text: str, use_bertweet: bool = True) -> Dict:
    """Return predicted act and emotion labels for a single utterance."""
    if use_bertweet:
        tok = bertweet_tokenizer
        a_model = act_bt_model
        e_model = emo_bt_model
    else:
        tok = act_tokenizer
        a_model = act_model
        e_model = emo_model

    enc = tok(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length",
    ).to(DEVICE)

    with torch.no_grad():
        act_logits = a_model(**enc).logits
        emo_logits = e_model(**enc).logits

    act_id = int(act_logits.argmax(dim=-1).item())
    emo_id = int(emo_logits.argmax(dim=-1).item())
    return {
        "act_id": act_id,
        "act_label": ACT_LABELS[act_id],
        "emo_id": emo_id,
        "emo_label": EMO_LABELS[emo_id],
        "act_probs": torch.softmax(act_logits, dim=-1).cpu().numpy().tolist()[0],
        "emo_probs": torch.softmax(emo_logits, dim=-1).cpu().numpy().tolist()[0],
    }


# Quick sanity check
for sample in [
    "How are you doing today?",
    "That makes me so angry!",
    "I can't believe you did that, no cap",
]:
    result = predict_act_emotion(sample)
    print(f"  {sample!r}")
    print(f"    -> act={result['act_label']}, emotion={result['emo_label']}")

  'How are you doing today?'
    -> act=question, emotion=no_emotion
  'That makes me so angry!'
    -> act=inform, emotion=anger
  "I can't believe you did that, no cap"
    -> act=inform, emotion=anger


---
## Part 3 — Gen Z Translation Dataset Build

Combine the local Gen Z CSV with the HuggingFace slang dataset
and normalize into bidirectional translation pairs.

In [13]:
from datasets import load_dataset

# ── Local CSV ────────────────────────────────────────────────────
genz_local_df = pd.read_csv(GENZ_CSV)
print("Local Gen Z CSV shape:", genz_local_df.shape)
print("Columns:", list(genz_local_df.columns))
display(genz_local_df.head(3))

# ── HuggingFace dataset ─────────────────────────────────────────
hf_ds = load_dataset(HF_GENZ_DATASET)
hf_df = pd.DataFrame(hf_ds["train"])
print("\nHF Gen Z dataset shape:", hf_df.shape)
print("Columns:", list(hf_df.columns))
display(hf_df.head(3))

Local Gen Z CSV shape: (229, 2)
Columns: ['Gen Z', 'Regular English']


,Gen Z,Regular English
0,I can't believe I rawdogged that exam.,I can't believe I went into that exam without ...
1,"No cap, This guy is highkey excited today.","Honestly, this guy is really excited today."
2,He is bussin right now.,He's doing really well right now.


README.md: 0.00B [00:00, ?B/s]

all_slangs.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1779 [00:00<?, ? examples/s]


HF Gen Z dataset shape: (1779, 4)
Columns: ['Slang', 'Description', 'Example', 'Context']


,Slang,Description,Example,Context
0,W,Shorthand for win,"Got the job today, big W!",Typically used in conversations to celebrate s...
1,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.",Often used when referring to a failure or mish...
2,L+ratio,Response to a comment or action on the interne...,Your tweet got 5 likes and 100 replies calling...,Popularized on social media platforms to signi...


In [14]:
# ── Normalize into bidirectional pairs ────────────────────────────
pairs = []

# From local CSV
if "Gen Z" in genz_local_df.columns and "Regular English" in genz_local_df.columns:
    for _, row in genz_local_df.iterrows():
        gz = str(row["Gen Z"]).strip()
        en = str(row["Regular English"]).strip()
        if gz and en:
            pairs.append({"gen_z": gz, "plain": en, "source": "local_csv"})

# From HF dataset
for _, row in hf_df.iterrows():
    slang = str(row.get("slang", "")).strip()
    desc  = str(row.get("description", "")).strip()
    example = str(row.get("example", "")).strip()
    if slang and desc:
        pairs.append({"gen_z": slang, "plain": desc, "source": "hf_lexicon"})
    if example and desc:
        pairs.append({"gen_z": example, "plain": desc, "source": "hf_example"})

parallel_df = (
    pd.DataFrame(pairs)
    .drop_duplicates(subset=["gen_z", "plain"])
    .reset_index(drop=True)
)
print(f"Total parallel pairs: {len(parallel_df)}")
print(f"Sources: {parallel_df['source'].value_counts().to_dict()}")

# ── Create bidirectional dataset ─────────────────────────────────
bidir_rows = []
for _, row in parallel_df.iterrows():
    bidir_rows.append({
        "input_text": "translate to Gen Z: " + row["plain"],
        "target_text": row["gen_z"],
        "direction": "standard_to_genz",
        "plain": row["plain"],
        "gen_z": row["gen_z"],
    })
    bidir_rows.append({
        "input_text": "translate to standard English: " + row["gen_z"],
        "target_text": row["plain"],
        "direction": "genz_to_standard",
        "plain": row["plain"],
        "gen_z": row["gen_z"],
    })

trans_df = pd.DataFrame(bidir_rows)
print(f"Bidirectional rows: {len(trans_df)}")

# ── Split ────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

train_trans_df, val_trans_df = train_test_split(
    trans_df, test_size=0.15, random_state=SEED
)
print(f"Translation train: {len(train_trans_df)}, val: {len(val_trans_df)}")

Total parallel pairs: 229
Sources: {'local_csv': 229}
Bidirectional rows: 458
Translation train: 389, val: 69


---
## Part 4 — Gen Z Translation with BART

Fine-tune `facebook/bart-base` for bidirectional
Gen Z ↔ standard English translation.

In [19]:
bart_tokenizer = AutoTokenizer.from_pretrained(CONFIG["bart_checkpoint"])
bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    CONFIG["bart_checkpoint"]
).to(DEVICE)


def tokenize_translation(batch):
    model_inputs = bart_tokenizer(
        batch["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length",
    )
    labels = bart_tokenizer(
        batch["target_text"],
        max_length=128,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_trans_ds = (
    Dataset.from_pandas(train_trans_df[["input_text", "target_text"]])
    .map(tokenize_translation, batched=True)
)
val_trans_ds = (
    Dataset.from_pandas(val_trans_df[["input_text", "target_text"]])
    .map(tokenize_translation, batched=True)
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=bart_tokenizer, model=bart_model
)

bart_args = Seq2SeqTrainingArguments(
    output_dir="./bart_genz_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["translation_lr"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["translation_epochs"],
    predict_with_generate=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
)

bart_trainer = Seq2SeqTrainer(
    model=bart_model,
    args=bart_args,
    train_dataset=train_trans_ds,
    eval_dataset=val_trans_ds,
    data_collator=data_collator
)

bart_trainer.train()
print("BART training complete.")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/389 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,3.650620
2,5.599455,1.436913
3,5.599455,0.364765
4,0.719616,0.105214
5,0.719616,0.053982
6,0.077584,0.038266
7,0.077584,0.033007
8,0.030848,0.032027


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


BART training complete.


In [20]:
def translate_bart(text: str, direction: str = "standard_to_genz") -> str:
    """Translate a single string using the fine-tuned BART model."""
    prefix = (
        "translate to Gen Z: "
        if direction == "standard_to_genz"
        else "translate to standard English: "
    )
    inputs = bart_tokenizer(
        prefix + text,
        return_tensors="pt",
        max_length=128,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        out = bart_model.generate(**inputs, max_new_tokens=64, num_beams=4)

    return bart_tokenizer.decode(out[0], skip_special_tokens=True)


# Quick test
for phrase in [
    "That outfit is fire no cap",
    "I'm feeling really happy today",
]:
    print(f"  {phrase!r}")
    print(f"    -> Gen Z:     {translate_bart(phrase, 'standard_to_genz')}")
    print(f"    -> Standard:  {translate_bart(phrase, 'genz_to_standard')}")

  'That outfit is fire no cap'
    -> Gen Z:     That outfit is definitely not that great
    -> Standard:  That outfit is definitely not that great
  "I'm feeling really happy today"
    -> Gen Z:     I am highkey happy today.
    -> Standard:  I am highkey happy today.


---
## Part 5 — Translation Evaluation

Automatic metrics (ROUGE, BERTScore) plus qualitative
examples in both directions.

In [23]:
import evaluate
from bert_score import score as bertscore_fn

rouge = evaluate.load("rouge")


def evaluate_translations(pred_df: pd.DataFrame) -> Dict[str, float]:
    predictions = pred_df["prediction"].fillna("").astype(str).tolist()
    references  = pred_df["reference"].fillna("").astype(str).tolist()
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,
    )
    P, R, F1 = bertscore_fn(predictions, references, lang="en", verbose=False)
    return {
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "bertscore_f1": float(F1.mean().item()),
    }


# Build prediction tables
eval_rows = []
for direction in ["standard_to_genz", "genz_to_standard"]:
    subset = val_trans_df[val_trans_df["direction"] == direction].sample(
        n=min(
            CONFIG["translation_eval_samples"],
            len(val_trans_df[val_trans_df["direction"] == direction]),
        ),
        random_state=SEED,
    ).reset_index(drop=True)
    for _, row in subset.iterrows():
        source = row["plain"] if direction == "standard_to_genz" else row["gen_z"]
        reference = row["gen_z"] if direction == "standard_to_genz" else row["plain"]
        eval_rows.append({
            "direction": direction,
            "source": source,
            "reference": reference,
            "prediction": translate_bart(source, direction),
        })

eval_df = pd.DataFrame(eval_rows)

metrics_rows = []
for direction, group in eval_df.groupby("direction"):
    metrics_rows.append({"direction": direction, **evaluate_translations(group)})

metrics_df = pd.DataFrame(metrics_rows)
print("\n=== Translation Metrics ===")
display(metrics_df)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Translation Metrics ===


,direction,rouge1,rouge2,rougeL,bertscore_f1
0,genz_to_standard,0.847797,0.712839,0.848213,0.977895
1,standard_to_genz,0.884591,0.852742,0.886359,0.980346


In [24]:
# ── Print example translations in both directions ────────────────
def print_translation_examples(df: pd.DataFrame, n: int = 5):
    for direction in ["standard_to_genz", "genz_to_standard"]:
        print("=" * 90)
        print("DIRECTION:", direction)
        subset = df[df["direction"] == direction].head(n)
        for _, row in subset.iterrows():
            print("-" * 90)
            print("SOURCE:    ", row["source"])
            print("REFERENCE: ", row["reference"])
            print("PREDICTED: ", row["prediction"])


print_translation_examples(eval_df, n=5)

DIRECTION: standard_to_genz
------------------------------------------------------------------------------------------
SOURCE:     I'm definitely really good.
REFERENCE:  Highkey, I am fire for real.
PREDICTED:  Highkey, I am fire tbh.
------------------------------------------------------------------------------------------
SOURCE:     That girl has definitely been very charismatic lately.
REFERENCE:  Highkey, That girl is full of rizz lately.
PREDICTED:  Highkey, That girl is full of rizz lately.
------------------------------------------------------------------------------------------
SOURCE:     My friend is coming across as a little strange today.
REFERENCE:  My friend is giving weird vibes today.
PREDICTED:  Lowkey, My friend is giving weird vibes today.
------------------------------------------------------------------------------------------
SOURCE:     To be honest, that girl is definitely really excited.
REFERENCE:  Highkey, That girl is highkey excited tbh.
PREDICTED:  Highk

---
## Part 6 — Integrated Pipeline: Translation Candidate Reranking
via Action/Emotion Consistency

Generate multiple translation candidates and rerank them using
downstream action/emotion consistency with the source utterance.

In [27]:
def generate_candidates(text: str, direction: str, n: int = 8) -> List[str]:
    """Generate multiple diverse translation candidates."""
    prefix = (
        "translate to Gen Z: "
        if direction == "standard_to_genz"
        else "translate to standard English: "
    )
    inputs = bart_tokenizer(
        prefix + text,
        return_tensors="pt",
        max_length=128,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = bart_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=max(n, 8),
            num_return_sequences=n,
            diversity_penalty=0.8,
            num_beam_groups=min(n, 4),
            no_repeat_ngram_size=3,
            trust_remote_code=True, # Added this line
        )

    candidates = [
        bart_tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]
    # Deduplicate while preserving order
    return list(dict.fromkeys(candidates))


def score_candidate_consistency(source: str, candidate: str) -> Dict:
    """Score how well a candidate preserves the source action/emotion profile."""
    src_profile  = predict_act_emotion(source)
    cand_profile = predict_act_emotion(candidate)

    act_match = 1.0 if src_profile["act_id"] == cand_profile["act_id"] else 0.0
    emo_match = 1.0 if src_profile["emo_id"] == cand_profile["emo_id"] else 0.0

    # Soft consistency: dot product on probability distributions
    act_sim = float(np.dot(src_profile["act_probs"], cand_profile["act_probs"]))
    emo_sim = float(np.dot(src_profile["emo_probs"], cand_profile["emo_probs"]))

    return {
        "act_match": act_match,
        "emo_match": emo_match,
        "act_soft_sim": act_sim,
        "emo_soft_sim": emo_sim,
        "consistency_score": (
            0.4 * act_sim + 0.4 * emo_sim
            + 0.1 * act_match + 0.1 * emo_match
        ),
        "src_act": src_profile["act_label"],
        "src_emo": src_profile["emo_label"],
        "cand_act": cand_profile["act_label"],
        "cand_emo": cand_profile["emo_label"],
    }


def rerank_candidates(
    source: str, direction: str, n_candidates: int = 8
) -> pd.DataFrame:
    """Generate candidates, score them, return ranked results."""
    candidates = generate_candidates(source, direction, n=n_candidates)
    rows = []
    for cand in candidates:
        scores = score_candidate_consistency(source, cand)
        rows.append({"candidate": cand, **scores})
    df = pd.DataFrame(rows).sort_values(
        "consistency_score", ascending=False
    ).reset_index(drop=True)
    return df

In [28]:
# ── Demo: reranking pipeline ─────────────────────────────────────
demo_inputs = [
    ("No cap, he is lowkey tired for real.", "genz_to_standard"),
    ("I feel really happy and excited today!", "standard_to_genz"),
    ("That's cringe bro, you're being delulu", "genz_to_standard"),
    ("She did an excellent job on the presentation.", "standard_to_genz"),
]

for source, direction in demo_inputs:
    print("=" * 90)
    print(f"SOURCE ({direction}): {source}")
    src_profile = predict_act_emotion(source)
    print(f"  Source act={src_profile['act_label']}, emo={src_profile['emo_label']}")

    ranked = rerank_candidates(source, direction)
    best = ranked.iloc[0]
    print(f"  Top candidate: {best['candidate']}")
    print(
        f"    -> act={best['cand_act']}, emo={best['cand_emo']}, "
        f"score={best['consistency_score']:.3f}"
    )
    print("  All candidates:")
    for _, row in ranked.iterrows():
        print(
            f"    [{row['consistency_score']:.3f}] {row['candidate']}  "
            f"(act={row['cand_act']}, emo={row['cand_emo']})"
        )
    print()

SOURCE (genz_to_standard): No cap, he is lowkey tired for real.
  Source act=inform, emo=no_emotion


generate.py: 0.00B [00:00, ?B/s]

beam_search.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/generate.py
- beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Passing `generation_config` together with generation-related arguments=({'diversity_penalty', 'num_beam_groups', 'num_beams', 'max_new_tokens', 'no_repeat_ngram_size', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64

  Top candidate: To be honest, he's definitely a little tired.
    -> act=inform, emo=no_emotion, score=0.978
  All candidates:
    [0.978] To be honest, he's definitely a little tired.  (act=inform, emo=no_emotion)
    [0.978] To be honest, he's definitely a little tired for real.  (act=inform, emo=no_emotion)
    [0.968] Honestly, he's a little tired for real.  (act=inform, emo=no_emotion)
    [0.965] Honestly, he's a little tired.  (act=inform, emo=no_emotion)
    [0.937] Honestly, he's kind of a little tired.  (act=inform, emo=no_emotion)
    [0.875] Honestly, he's definitely a little tired.  (act=inform, emo=no_emotion)

SOURCE (standard_to_genz): I feel really happy and excited today!
  Source act=inform, emo=happiness


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Top candidate: Lowkey, I am highkey excited today.
    -> act=inform, emo=happiness, score=0.840
  All candidates:
    [0.840] Lowkey, I am highkey excited today.  (act=inform, emo=happiness)
    [0.834] I am highkey excited today.  (act=inform, emo=happiness)
    [0.826] Highkey, I am highkey excited today.  (act=inform, emo=happiness)

SOURCE (genz_to_standard): That's cringe bro, you're being delulu
  Source act=inform, emo=anger


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Top candidate: That is embarrassing, you're delulu
    -> act=inform, emo=anger, score=0.664
  All candidates:
    [0.664] That is embarrassing, you're delulu  (act=inform, emo=anger)
    [0.664] That is a little embarrassing, you're delulu  (act=inform, emo=anger)
    [0.548] Seriously, you're delulu  (act=inform, emo=no_emotion)
    [0.547] That is a little strange, you're delulu  (act=inform, emo=no_emotion)
    [0.547] Honestly, you're being unrealistic.  (act=inform, emo=no_emotion)
    [0.546] That is delulu, you're delulu  (act=inform, emo=no_emotion)
    [0.545] Seriously, you're delulu.  (act=inform, emo=no_emotion)
    [0.544] Honestly, you're being a little unrealistic  (act=inform, emo=no_emotion)

SOURCE (standard_to_genz): She did an excellent job on the presentation.
  Source act=inform, emo=happiness
  Top candidate: She did a great job on the presentation.
    -> act=inform, emo=happiness, score=0.864
  All candidates:
    [0.864] She did a great job on the presentat

---
## Part 7 — End-to-End Demo

Full pipeline: Gen Z input → translate → predict action/emotion → summarize.

In [29]:
def end_to_end(text: str, direction: str = "genz_to_standard") -> None:
    """Run the full pipeline on one input."""
    print("=" * 90)
    print(f"INPUT: {text}")
    print(f"DIRECTION: {direction}")

    # Source behavioral profile
    src = predict_act_emotion(text)
    print(f"\nSource profile: act={src['act_label']}, emotion={src['emo_label']}")

    # Generate & rerank candidates
    ranked = rerank_candidates(text, direction)
    best = ranked.iloc[0]

    print(f"\nBest translation: {best['candidate']}")
    print(f"  Consistency score: {best['consistency_score']:.3f}")
    print(f"  Candidate act={best['cand_act']}, emotion={best['cand_emo']}")

    # Summary
    preserved = (
        "YES"
        if best["act_match"] == 1.0 and best["emo_match"] == 1.0
        else "PARTIAL"
        if best["act_match"] + best["emo_match"] >= 1.0
        else "NO"
    )
    print(f"\n  Behavioral preservation: {preserved}")
    print(
        f"  The source was classified as [{src['act_label']}]"
        f" with [{src['emo_label']}] emotion."
    )
    print(
        f"  The best translation was classified as [{best['cand_act']}]"
        f" with [{best['cand_emo']}] emotion."
    )
    print()


# Run demos
end_to_end("No cap, this guy is highkey excited today.", "genz_to_standard")
end_to_end("I'm honestly really tired and a bit annoyed.", "standard_to_genz")
end_to_end("She ate and left no crumbs", "genz_to_standard")
end_to_end("That was very embarrassing and awkward.", "standard_to_genz")

Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INPUT: No cap, this guy is highkey excited today.
DIRECTION: genz_to_standard

Source profile: act=inform, emotion=no_emotion


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Best translation: To be honest, this guy is really excited today.
  Consistency score: 0.888
  Candidate act=inform, emotion=no_emotion

  Behavioral preservation: YES
  The source was classified as [inform] with [no_emotion] emotion.
  The best translation was classified as [inform] with [no_emotion] emotion.

INPUT: I'm honestly really tired and a bit annoyed.
DIRECTION: standard_to_genz

Source profile: act=inform, emotion=no_emotion


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Best translation: Highkey, I am lowkey tired and a bit tired for real.
  Consistency score: 0.946
  Candidate act=inform, emotion=no_emotion

  Behavioral preservation: YES
  The source was classified as [inform] with [no_emotion] emotion.
  The best translation was classified as [inform] with [no_emotion] emotion.

INPUT: She ate and left no crumbs
DIRECTION: genz_to_standard

Source profile: act=inform, emotion=no_emotion


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Best translation: To be honest, she ate and left no crumbs.
  Consistency score: 0.972
  Candidate act=inform, emotion=no_emotion

  Behavioral preservation: YES
  The source was classified as [inform] with [no_emotion] emotion.
  The best translation was classified as [inform] with [no_emotion] emotion.

INPUT: That was very embarrassing and awkward.
DIRECTION: standard_to_genz

Source profile: act=inform, emotion=anger

Best translation: That is embarrassing and awkward.
  Consistency score: 0.679
  Candidate act=inform, emotion=anger

  Behavioral preservation: YES
  The source was classified as [inform] with [anger] emotion.
  The best translation was classified as [inform] with [anger] emotion.



---
## Part 8 — Research Discussion & Next Steps

**Key findings:**
- BERT and BERTweet baselines for action/emotion classification are compared;
  BERTweet should handle informal text better
- BART-based translation supports bidirectional Gen Z ↔ standard English
- The reranking pipeline introduces a novel behavioral-consistency constraint
  on translation

**Future work:**
- Experiment with temporal ambiguity handling ("I'll do it later", "see u tmr")
- Scale the Gen Z dataset with data augmentation
- Try T5-based translation as a second baseline for direct comparison
- Explore ensemble reranking with multiple consistency signals